In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.io as pio
import kairo

pio.renderers.default = "notebook_connected"
pd.set_option("display.precision", 3)

LOG = "../../data/thesis/general/running_example/L3.xes"
META = ["case:concept:name", "time:timestamp"]
CONTROL_FLOW = pd.Timestamp("2020-04-01 06:00", tz="UTC")

In [2]:
log = kairo.read_log(LOG, resource="org:resource", event_duration="event:duration_min")
print(log["case:concept:name"].nunique(), "cases,", len(log), "events")

4920 cases, 24600 events


In [3]:
running_cases = log[log["case:concept:name"].isin(["case_000001", "case_003711"])]
running_cases[["case:concept:name", "concept:name", "time:timestamp", "org:resource", "event_duration", "case:amount", "case:region"]]

,case:concept:name,concept:name,time:timestamp,org:resource,event_duration,case:amount,case:region
0,case_000001,a,2020-01-01 00:17:34+00:00,res_01,17.567,780.51,region_4
1,case_000001,b,2020-01-01 00:49:19+00:00,res_01,27.983,780.51,region_4
2,case_000001,c,2020-01-01 01:40:48+00:00,res_02,39.917,780.51,region_4
3,case_000001,d,2020-01-01 02:16:29+00:00,res_01,19.217,780.51,region_4
4,case_000001,e,2020-01-01 02:57:12+00:00,res_04,25.817,780.51,region_4
18550,case_003711,a,2020-10-02 01:07:21+00:00,res_03,26.250,2476.48,region_4
18551,case_003711,c,2020-10-02 01:49:46+00:00,res_03,31.483,2476.48,region_4
18552,case_003711,b,2020-10-02 02:19:29+00:00,res_02,28.483,2476.48,region_4
18553,case_003711,e,2020-10-02 03:00:18+00:00,res_02,28.417,2476.48,region_4
18554,case_003711,d,2020-10-02 03:45:02+00:00,res_02,27.783,2476.48,region_4


In [4]:
features = kairo.compute_features_intra(log, features=["act_freqs", "df_counts", "current_act"])
print(features.shape[1] - 2, "feature columns")

20 feature columns


In [5]:
third_events = features.groupby("case:concept:name").nth(2)
third_events[third_events["case:concept:name"].isin(["case_000001", "case_003711"])].set_index("case:concept:name").drop(columns="time:timestamp").T

case:concept:name,case_000001,case_003711
act_freqs:a,0.333,0.333
act_freqs:b,0.333,0.333
act_freqs:c,0.333,0.333
act_freqs:d,0.000,0.000
act_freqs:e,0.000,0.000
df_counts:a -> b,1.000,0.000
df_counts:a -> c,0.000,1.000
df_counts:b -> c,1.000,0.000
df_counts:b -> d,0.000,0.000
df_counts:b -> e,0.000,0.000


In [6]:
scaled = kairo.standardize(features, method="minmax")
features.drop(columns=META).agg(["min", "max"]).T

,min,max
act_freqs:a,0.2,1.00
act_freqs:b,0.0,0.50
act_freqs:c,0.0,0.50
act_freqs:d,0.0,0.25
act_freqs:e,0.0,0.25
df_counts:a -> b,0.0,1.00
df_counts:a -> c,0.0,1.00
df_counts:b -> c,0.0,1.00
df_counts:b -> d,0.0,1.00
df_counts:b -> e,0.0,1.00


In [7]:
pca = kairo.compute_pca(scaled)
errors = kairo.compute_pca_reconstruction_errors(scaled, pca)
values = scaled.drop(columns=META)
print(len(values.drop_duplicates()), "distinct rows")
print("error without components", round(((values - values.mean()) ** 2).sum(axis=1).mean(), 3))
pd.DataFrame({
    "explained variance %": 100 * pca.explained_variance_ratio_,
    "cumulative %": 100 * pca.explained_variance_ratio_.cumsum(),
    "reconstruction error": errors.to_numpy(),
}, index=errors.index).head(11)

13 distinct rows
error without components 2.848


,explained variance %,cumulative %,reconstruction error
components,,,
1,2.867e+01,28.668,2.031
2,2.183e+01,50.502,1.409
3,1.385e+01,64.354,1.015
4,1.068e+01,75.036,0.711
5,1.043e+01,85.466,0.414
6,6.364e+00,91.830,0.233
7,3.041e+00,94.871,0.146
8,2.129e+00,97.000,0.085
9,1.820e+00,98.821,0.034


In [8]:
cut = 6
kairo.plot_pca_variances(pca, cut_component=cut).show()
kairo.plot_pca_reconstruction_errors(errors, cut_component=cut).show()

In [9]:
loadings = pd.DataFrame(pca.components_[:2].T, index=pca.feature_names_in_, columns=["pc_1", "pc_2"])
for column in loadings:
    print(column, loadings[column].reindex(loadings[column].abs().sort_values(ascending=False).index).head(4).round(2).to_dict())

pc_1 {'df_counts:b -> c': 0.48, 'df_counts:a -> b': 0.45, 'df_counts:c -> d': 0.3, 'act_freqs:a': -0.3}
pc_2 {'df_counts:a -> c': 0.5, 'df_counts:c -> b': 0.47, 'df_counts:a -> b': -0.28, 'act_freqs:e': 0.26}


In [10]:
df_counts = features.filter(like="df_counts")
pd.DataFrame({
    "share of prefixes": df_counts.gt(0).mean(),
    "z-score of a count of 1": kairo.standardize(features, method="zscore").filter(like="df_counts").max(),
})

,share of prefixes,z-score of a count of 1
df_counts:a -> b,0.502,0.995
df_counts:a -> c,0.298,1.536
df_counts:b -> c,0.377,1.286
df_counts:b -> d,0.076,3.496
df_counts:b -> e,0.073,3.557
df_counts:c -> b,0.223,1.865
df_counts:c -> d,0.178,2.151
df_counts:c -> e,0.073,3.553
df_counts:d -> e,0.127,2.626
df_counts:e -> d,0.073,3.555


In [11]:
compressed = kairo.apply_pca(scaled, pca, cut_component=cut)
compressed.head()

,case:concept:name,time:timestamp,pc_1,pc_2,pc_3,pc_4,pc_5,pc_6
0,case_000001,2020-01-01 00:17:34+00:00,-1.111,-0.672,0.804,0.060,-0.039,0.023
1,case_000001,2020-01-01 00:49:19+00:00,-0.168,-0.606,-0.731,-0.725,-0.688,0.008
2,case_000001,2020-01-01 01:40:48+00:00,0.501,-0.698,-0.721,0.335,0.641,0.063
3,case_000001,2020-01-01 02:16:29+00:00,1.248,-0.212,0.168,0.740,-0.655,-0.449
4,case_000001,2020-01-01 02:57:12+00:00,1.537,0.109,0.725,-0.668,0.347,-0.412


In [12]:
scores = {}
for size in [(2, 2), (3, 3), (4, 4), (5, 5)]:
    som = kairo.compute_som(compressed, size=size)
    states = kairo.get_som_states(compressed, som)
    scores[f"{size[0]}x{size[1]}"] = {
        "used cells": len(states.drop_duplicates()),
        "quantization error": kairo.compute_som_quantization_error(compressed, states, som),
        "occupancy entropy": kairo.compute_som_occupancy_entropy(states, size),
        "silhouette": kairo.compute_silhouette(compressed, states),
        "Calinski-Harabasz": kairo.compute_calinski_harabasz(compressed, states),
    }
pd.DataFrame(scores).T

,used cells,quantization error,occupancy entropy,silhouette,Calinski-Harabasz
2x2,4.0,0.917,0.993,0.527,13112.671
3x3,9.0,0.288,0.973,0.857,36693.600
4x4,11.0,0.109,0.826,0.948,109548.401
5x5,12.0,0.052,0.727,0.975,218726.031


In [13]:
size = (3, 3)
som = kairo.compute_som(compressed, size=size)
states = pd.concat([compressed, kairo.get_som_states(compressed, som)], axis=1)

In [14]:
ordered = log.sort_values("time:timestamp", kind="stable")
prefixes = ordered.groupby("case:concept:name")["concept:name"].transform(lambda acts: acts.cumsum())

cells = states.assign(
    cell=[f"({i + 1}, {j + 1})" for i, j in zip(states["i"], states["j"])],
    prefix=prefixes,
    period=np.where(states["time:timestamp"] < CONTROL_FLOW, "before", "after"),
)
print(cells["period"].value_counts().to_dict())
cells.groupby(["cell", "prefix", "period"]).size().unstack(fill_value=0)[["before", "after"]]

{'after': 18345, 'before': 6255}


period         before  after
cell   prefix               
(1, 1) a         1254   3666
(1, 2) acbd         0    930
       acbed        0    901
(1, 3) acbde        0    930
       acbe         0    901
(2, 1) ab        1253   1836
(2, 2) acb          0   1831
(2, 3) ac           0   1831
(3, 1) abcde     1249    937
       abce         0    903
(3, 2) abcd      1249    937
       abced        0    903
(3, 3) abc       1250   1839

In [15]:
shares = cells.groupby(["period", "cell"]).size() / cells.groupby("period").size()
(100 * shares).unstack(level=0).fillna(0)[["before", "after"]].round(1)

period,before,after
cell,,
"(1, 1)",20.0,20.0
"(1, 2)",0.0,10.0
"(1, 3)",0.0,10.0
"(2, 1)",20.0,10.0
"(2, 2)",0.0,10.0
"(2, 3)",0.0,10.0
"(3, 1)",20.0,10.0
"(3, 2)",20.0,10.0
"(3, 3)",20.0,10.0


In [16]:
for case in ["case_000001", "case_003711"]:
    print(case, cells[cells["case:concept:name"] == case]["cell"].tolist())

case_000001 ['(1, 1)', '(2, 1)', '(3, 3)', '(3, 2)', '(3, 1)']
case_003711 ['(1, 1)', '(2, 3)', '(2, 2)', '(1, 3)', '(1, 2)']


In [17]:
kairo.plot_som_heatmap(states, size=size, end_date=CONTROL_FLOW).show()
kairo.plot_som_heatmap(states, size=size, start_date=CONTROL_FLOW).show()

In [18]:
cell_states = states[["i", "j"]]
pd.Series({
    "states": len(cell_states.drop_duplicates()),
    "silhouette": kairo.compute_silhouette(compressed, cell_states),
    "Calinski-Harabasz": kairo.compute_calinski_harabasz(compressed, cell_states),
    "quantization error": kairo.compute_som_quantization_error(compressed, cell_states, som),
    "occupancy entropy": kairo.compute_som_occupancy_entropy(cell_states, size),
})

states                    9.000
silhouette                0.857
Calinski-Harabasz     36693.600
quantization error        0.288
occupancy entropy         0.973
dtype: float64

In [19]:
state_colors = kairo.get_som_state_colors(som)

for case in ["case_000001", "case_003711"]:
    trajectory = kairo.get_som_case_trajectory(states, case)
    first = trajectory["start"].min()
    print(case)
    print(trajectory.assign(
        cell=[f"({int(c[1]) + 1}, {int(c[4]) + 1})" for c in trajectory["state"]],
        minutes_from=(trajectory["start"] - first).dt.total_seconds() / 60,
        minutes_to=(trajectory["end"] - first).dt.total_seconds() / 60,
        color=[state_colors[(int(c[1]), int(c[4]))] for c in trajectory["state"]],
    )[["cell", "minutes_from", "minutes_to", "color"]].round(2).to_string(), "\n")
    kairo.plot_som_case_trajectory(trajectory, state_colors, case_id=case).show()

case_000001
     cell  minutes_from  minutes_to    color
0  (1, 1)          0.00       31.75  #009eb4
1  (2, 1)         31.75       83.23  #58a7b0
2  (3, 3)         83.23      118.92  #8d9c4a
3  (3, 2)        118.92      159.63  #c98b55
4  (3, 1)        159.63      159.63  #db929c 



case_003711
     cell  minutes_from  minutes_to    color
0  (1, 1)          0.00       42.42  #009eb4
1  (2, 3)         42.42       72.13  #547959
2  (2, 2)         72.13      112.95  #527491
3  (1, 3)        112.95      157.68  #955ca3
4  (1, 2)        157.68      157.68  #855a6e 



In [20]:
distances = kairo.get_som_state_distances(som)
names = [f"({int(c[1]) + 1}, {int(c[4]) + 1})" for c in distances.index]
distances.index, distances.columns = names, names
off_diagonal = distances.to_numpy()[~np.eye(len(distances), dtype=bool)]
print("smallest", off_diagonal.min().round(2), "largest", off_diagonal.max().round(2), "mean", off_diagonal.mean().round(2))

for case in ["case_000001", "case_003711"]:
    path = cells[cells["case:concept:name"] == case]["cell"].tolist()
    print(case, [round(distances.loc[a, b], 2) for a, b in zip(path, path[1:])])
distances.round(2)

smallest 1.56 largest 2.78 mean 2.23
case_000001 [np.float64(2.0), np.float64(1.8), np.float64(1.77), np.float64(1.7)]
case_003711 [np.float64(1.97), np.float64(1.63), np.float64(1.72), np.float64(1.67)]


,"(1, 1)","(1, 2)","(1, 3)","(2, 1)","(2, 2)","(2, 3)","(3, 1)","(3, 2)","(3, 3)"
"(1, 1)",0.00,2.49,2.61,2.00,2.24,1.97,2.71,2.58,2.32
"(1, 2)",2.49,0.00,1.67,2.53,1.56,2.01,2.76,2.14,2.67
"(1, 3)",2.61,1.67,0.00,2.63,1.72,2.11,2.21,2.73,2.76
"(2, 1)",2.00,2.53,2.63,0.00,1.63,2.30,2.33,2.17,1.80
"(2, 2)",2.24,1.56,1.72,1.63,0.00,1.63,2.78,2.62,2.26
"(2, 3)",1.97,2.01,2.11,2.30,1.63,0.00,2.70,2.54,1.60
"(3, 1)",2.71,2.76,2.21,2.33,2.78,2.70,0.00,1.70,1.99
"(3, 2)",2.58,2.14,2.73,2.17,2.62,2.54,1.70,0.00,1.77
"(3, 3)",2.32,2.67,2.76,1.80,2.26,1.60,1.99,1.77,0.00


In [21]:
kairo.plot_som_u_matrix(som).show()
kairo.plot_som_state_colors(states, state_colors).show()

In [22]:
u_matrix = pd.DataFrame(som.distance_map(scaling="mean"), index=[1, 2, 3], columns=[1, 2, 3])
new_cells = [(1, 2), (1, 3), (2, 2), (2, 3)]
print("new cells", np.mean([u_matrix.loc[i, j] for i, j in new_cells]).round(2),
      "old cells", np.mean([u_matrix.loc[i, j] for i in u_matrix.index for j in u_matrix.columns if (i, j) not in new_cells]).round(2))
u_matrix.round(2)

new cells 0.87 old cells 0.94


,1,2,3
1,0.99,0.90,0.81
2,0.94,0.90,0.87
3,1.00,0.95,0.83


In [23]:
distributions = kairo.compute_state_distributions(states, signal_window="1D")
events = states.groupby(states["time:timestamp"].dt.floor("1D")).size()
print(len(distributions), "signal windows, events per signal window: median", events.median(), "lowest", events.min(), events.idxmin().date())

366 signal windows, events per signal window: median 66.0 lowest 3 2020-12-31


In [24]:
drift_signal = kairo.compute_drift_signal(distributions, divergence="kl", reference="recent", lookback=7)
scores = drift_signal.set_index("signal_window")["drift_score"]
print("highest before the drift", scores[:"2020-03-31"].max().round(3), "median after the drift", scores["2020-04-02":].median().round(3))
scores.nlargest(6).round(2)

highest before the drift 0.013 median after the drift 0.022


signal_window
2020-04-01 00:00:00+00:00    3.47
2020-12-31 00:00:00+00:00    1.15
2020-04-02 00:00:00+00:00    1.10
2020-05-01 00:00:00+00:00    0.68
2020-07-18 00:00:00+00:00    0.55
2020-04-03 00:00:00+00:00    0.50
Name: drift_score, dtype: float64

In [25]:
kairo.plot_drift_signal(drift_signal, divergence="kl", reference="recent").show()

In [26]:
previous = kairo.compute_drift_signal(distributions, divergence="kl", reference="previous").set_index("signal_window")["drift_score"]
print("highest before the drift", previous[:"2020-03-31"].max().round(3))
previous.nlargest(4).round(2)

highest before the drift 0.039


signal_window
2020-07-19 00:00:00+00:00    7.98
2020-05-02 00:00:00+00:00    5.52
2020-04-01 00:00:00+00:00    3.47
2020-12-31 00:00:00+00:00    1.02
Name: drift_score, dtype: float64

In [27]:
traces = log.sort_values("time:timestamp").groupby("case:concept:name").agg(trace=("concept:name", "".join), start=("time:timestamp", "min"))
days = pd.to_datetime(["2020-05-01", "2020-07-18"]).tz_localize("UTC")
for day in days:
    print(day.date(), traces[traces["start"].dt.floor("1D") == day]["trace"].value_counts().to_dict())
distributions.loc[days].round(3)

2020-05-01 {'abcde': 4, 'abced': 4}
2020-07-18 {'acbde': 4, 'acbed': 1}


state,"(0, 0)","(0, 1)","(0, 2)","(1, 0)","(1, 1)","(1, 2)","(2, 0)","(2, 1)","(2, 2)"
2020-05-01 00:00:00+00:00,0.222,0.0,0.0,0.194,0.0,0.0,0.194,0.194,0.194
2020-07-18 00:00:00+00:00,0.200,0.2,0.2,0.000,0.2,0.2,0.000,0.000,0.000


In [28]:
weekly = kairo.compute_state_distributions(states, signal_window="7D")
weekly_signal = kairo.compute_drift_signal(weekly, divergence="kl", reference="baseline")
weekly_scores = weekly_signal.set_index("signal_window")["drift_score"]
events = states.groupby(states["time:timestamp"].dt.floor("7D")).size()
print(len(weekly), "signal windows from", weekly.index[0].date(), weekly.index[0].day_name(), "events per signal window: median", events.median())
print("baseline", weekly.mean().round(3).to_dict())
period = np.select([weekly_scores.index < CONTROL_FLOW.floor("7D"), weekly_scores.index > CONTROL_FLOW], ["before", "after"], "drift")
print(weekly_scores.groupby(period).describe().round(3).to_string())
weekly_scores.loc["2020-03-19":"2020-04-09"].round(3)

54

 signal windows from 2019-12-26 Thursday events per signal window: median 472.0
baseline {'(0, 0)': 0.197, '(0, 1)': 0.072, '(0, 2)': 0.072, '(1, 0)': 0.124, '(1, 1)': 0.072, '(1, 2)': 0.072, '(2, 0)': 0.13, '(2, 1)': 0.13, '(2, 2)': 0.13}
        count   mean    std    min    25%    50%    75%    max
after    40.0  0.060  0.145  0.001  0.019  0.028  0.051  0.943
before   13.0  0.357  0.002  0.352  0.357  0.358  0.358  0.360
drift     1.0  0.232    NaN  0.232  0.232  0.232  0.232  0.232


signal_window
2020-03-19 00:00:00+00:00    0.357
2020-03-26 00:00:00+00:00    0.232
2020-04-02 00:00:00+00:00    0.082
2020-04-09 00:00:00+00:00    0.016
Name: drift_score, dtype: float64

In [29]:
weekly_scores["2020-04-02":"2020-12-24"].max().round(3), weekly_scores.tail(1).round(3), events.tail(1)

(np.float64(0.096),
 signal_window
 2020-12-31 00:00:00+00:00    0.943
 Name: drift_score, dtype: float64,
 time:timestamp
 2020-12-31 00:00:00+00:00    3
 dtype: int64)

In [30]:
kairo.plot_drift_signal(weekly_signal, divergence="kl", reference="baseline").show()

In [31]:
from thesis_figures import write_drift_signal

write_drift_signal("drift_signal_intra_1d", drift_signal, "1D", CONTROL_FLOW, y_max=4, y_ticks=[0, 1, 2, 3, 4])
write_drift_signal("drift_signal_intra_7d", weekly_signal, "7D", CONTROL_FLOW, y_max=1, y_ticks=[0, 0.5, 1])

['../../thesis/figures/tikz/drift_signal_intra_7d.tex',
 '../../thesis-de/figures/tikz/drift_signal_intra_7d.tex']

In [32]:
abstraction = kairo.abstract_features(features)
print(features.drop(columns=META).size, "feature values,", len(abstraction), "characters in the abstraction")
print(len(kairo.DEFAULT_SYSTEM_PROMPT), "characters in the system prompt")

492000 feature values, 1084 characters in the abstraction
11068 characters in the system prompt
